## Block 1: Imports & Setup

In [1]:
import time
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42

## Block 2: Load Dataset & Features

In [2]:
df = pd.read_csv("cleaned_dataset.csv")

raw_features = [
    "Log GDP per capita", "Social support", "Healthy life expectancy at birth",
    "Freedom to make life choices", "Generosity", "Perceptions of corruption",
    "Positive affect", "Negative affect"
]
target_col = "Life Ladder"  # continuous happiness score (regression target)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[raw_features])
X = pd.DataFrame(X_scaled, columns=raw_features)
y = df[target_col]

## Block 3: Train/Validation/Test Split

In [3]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE)
# no stratify here — Life Ladder is continuous, stratify only applies to classification labels
# random_state=RANDOM_STATE → ensures reproducibility (same split every run)

## Block 4: Define Models

In [4]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(max_depth=8, random_state=RANDOM_STATE),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=300, max_depth=12, random_state=RANDOM_STATE),
    "SVR": SVR(kernel="rbf"),
}

## Block 5: Train & Evaluate Models

In [5]:
rows = []
fitted = {}
for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start

    val_pred = model.predict(X_val)
    val_r2 = r2_score(y_val, val_pred)

    test_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, test_pred)
    mse = mean_squared_error(y_test, test_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, test_pred)

    rows.append({
        "Model": name,
        "MAE": round(mae, 4),
        "MSE": round(mse, 4),
        "RMSE": round(rmse, 4),
        "R2": round(r2, 4),
        "Training Time (s)": round(train_time, 3),
        "Val R2": round(val_r2, 4),
    })
    fitted[name] = model
    print(f"{name:24s} | test R2={r2:.4f}  val R2={val_r2:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}  train time={train_time:.3f}s")


Linear Regression        | test R2=0.7626  val R2=0.7637  RMSE=0.5320  MAE=0.4149  train time=0.059s
Decision Tree Regressor  | test R2=0.7740  val R2=0.8291  RMSE=0.5191  MAE=0.3898  train time=0.015s
Random Forest Regressor  | test R2=0.8610  val R2=0.8745  RMSE=0.4072  MAE=0.3055  train time=3.039s
SVR                      | test R2=0.8378  val R2=0.8134  RMSE=0.4398  MAE=0.3386  train time=0.157s


## Block 6: Results & Best Model

In [6]:
results_df = pd.DataFrame(rows).sort_values("Val R2", ascending=False).reset_index(drop=True)
print("\n", results_df.to_string(index=False))

best_name = results_df.iloc[0]["Model"]
best_model = fitted[best_name]
best_test_r2 = results_df.iloc[0]["R2"]
print(f"\nBest model (by validation R2): {best_name}")


                   Model    MAE    MSE   RMSE     R2  Training Time (s)  Val R2
Random Forest Regressor 0.3055 0.1658 0.4072 0.8610              3.039  0.8745
Decision Tree Regressor 0.3898 0.2695 0.5191 0.7740              0.015  0.8291
                    SVR 0.3386 0.1934 0.4398 0.8378              0.157  0.8134
      Linear Regression 0.4149 0.2831 0.5320 0.7626              0.059  0.7637

Best model (by validation R2): Random Forest Regressor


## Block 7: Prediction Sanity Check

In [7]:
print("\nSample predictions vs actual (test set, best model):")
sample_compare = pd.DataFrame({
    "Actual": y_test.values[:10],
    "Predicted": best_model.predict(X_test)[:10]
}).round(3)
print(sample_compare.to_string(index=False))



Sample predictions vs actual (test set, best model):
 Actual  Predicted
  6.685      6.399
  5.385      5.383
  4.277      4.503
  4.517      4.300
  5.590      5.689
  6.680      6.430
  7.181      7.120
  4.657      5.572
  3.483      4.176
  3.795      4.713


## Block 8: Save Model & Results

In [8]:
# R2/MAE/RMSE for every model, keyed by name (used by a model-select dropdown in the app, same pattern as HappyClassify)
test_r2_by_model = dict(zip(results_df["Model"], results_df["R2"]))
test_rmse_by_model = dict(zip(results_df["Model"], results_df["RMSE"]))
test_mae_by_model = dict(zip(results_df["Model"], results_df["MAE"]))

bundle = {
    "models": fitted,                  # ALL trained models: name -> fitted estimator
    "best_model_name": best_name,       # e.g. "Random Forest Regressor"
    "scaler": scaler,
    "features": raw_features,
    "target": target_col,
    "countries": sorted(df["Country name"].unique().tolist()),
    "test_r2": test_r2_by_model,        # dict: name -> test R2
    "test_rmse": test_rmse_by_model,    # dict: name -> test RMSE
    "test_mae": test_mae_by_model,      # dict: name -> test MAE
}
joblib.dump(bundle, "happypredict_model.pkl")
results_df.drop(columns=["Val R2"]).to_csv("model_comparison_regression.csv", index=False)
print(f"\nSaved happypredict_model.pkl (all {len(fitted)} models) and model_comparison_regression.csv")



Saved happypredict_model.pkl (all 4 models) and model_comparison_regression.csv
